In [4]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

# --- 1. CONFIGURAZIONE ---
# Inserisci qui il nome esatto della cartella dove hai salvato il tuo modello migliore
# (es. "bert_medico_full" oppure "bert_medico_10_shot")
CARTELLA_MODELLO = "model/bert_medico_full" 

print(f"Caricamento del modello da: {CARTELLA_MODELLO}...")

# Carichiamo il Tokenizer base e il TUO modello addestrato
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-italian-cased")
modello = AutoModelForTokenClassification.from_pretrained(CARTELLA_MODELLO)

# Creiamo la "Pipeline" (è uno strumento di Hugging Face che fa tutto il lavoro sporco 
# di tokenizzazione, predizione e ri-allineamento delle parole)
# aggregation_strategy="simple" unisce in automatico i sub-token ("elettro" + "##cardio")
ner_pipeline = pipeline("token-classification", model=modello, tokenizer=tokenizer, aggregation_strategy="simple")

# --- 2. IL TEST DAL VIVO ---
print("\nScrivi un referto medico inventato (o premi Invio per usare l'esempio).")
print("Digita 'esci' per terminare.")

while True:
    testo_input = input("\nReferto: ")
    
    if testo_input.lower() == 'esci':
        break
        
    if not testo_input.strip():
        # Frase di esempio se premi solo Invio
        testo_input = "Il paziente di 55 anni giunge in PS lamentando una grave dispnea e forte dolore toracico. Si prescrive tachipirina e riposo."
        print(f"Uso l'esempio: {testo_input}")

    # Chiediamo al modello di trovare le malattie!
    risultati = ner_pipeline(testo_input)
    
    print("\n--- RISULTATI ESTRATTI DA BERT FT ---")
    if not risultati:
        print("Nessuna entità clinica trovata.")
    else:
        for entita in risultati:
            parola = entita['word']
            etichetta = entita['entity_group']
            score = entita['score'] * 100
            
            # Stampiamo il risultato pulito
            print(f" Trovato: '{parola}'")
            print(f"   Tipo: {etichetta} (Score del modello: {score:.1f}%)\n")

Caricamento del modello da: model/bert_medico_full...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 912.48it/s, Materializing param=classifier.weight]                                      



Scrivi un referto medico inventato (o premi Invio per usare l'esempio).
Digita 'esci' per terminare.
Uso l'esempio: Il paziente di 55 anni giunge in PS lamentando una grave dispnea e forte dolore toracico. Si prescrive tachipirina e riposo.

--- RISULTATI ESTRATTI DA BERT FT ---
 Trovato: 'dispnea'
   Tipo: CLINENTITY (Score del modello: 93.9%)

 Trovato: 'dolore toracico'
   Tipo: CLINENTITY (Score del modello: 95.8%)

